# Chapter `1.3` - Memory

## Setup

### Module imports

In [31]:
from os import getenv
from dotenv import load_dotenv

from pprint import pprint
from typing import Dict, Any
from IPython.display import Markdown

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

from langchain.messages import AIMessage
from langchain.messages import HumanMessage

from langgraph.checkpoint.memory import InMemorySaver

### **Gemini** API setup

In [32]:
load_dotenv()

GOOGLE_API_KEY = getenv("GOOGLE_API_KEY")
GEMINI_API_MODEL = getenv("GEMINI_API_MODEL")

model = ChatGoogleGenerativeAI(model=GEMINI_API_MODEL, api_key=GOOGLE_API_KEY)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## Case `1`
- #### **No** STM (Short-Term Memory)

In [33]:
sys_prompt = "You are a friendly chatbot. Talk to the user like you are trying to get to know them better!"

agent = create_agent(
    model=model,
    system_prompt=sys_prompt
)

In [15]:
question1 = HumanMessage(content="Hi, my name is Nirmit and my favourite colour is blue.")
response = agent.invoke(
    {"messages": [question1]} 
)

Markdown(response["messages"][-1].content)

As a large language model, I don't have personal memories of past conversations or interactions with individual users. Therefore, I don't remember what your favourite colour is.

If you'd like to tell me your favourite colour, I'd be happy to remember it for our current conversation! 😊

In [9]:
question2 = HumanMessage(content="Do you remember what was my favourite colour?")

response = agent.invoke(
    {"messages": [question2]} 
)

Markdown(response["messages"][-1].content)

As a large language model, I don't have personal memories of past conversations or interactions with individual users. Therefore, I don't remember what your favourite colour is.

If you'd like to tell me your favourite colour, I'd be happy to remember it for our current conversation! 😊

## Case `2`
- #### Short-Term Memory (**Checkpoint**)

In [13]:
agent_stm = create_agent(
    model=model,
    system_prompt=sys_prompt,
    checkpointer=InMemorySaver(), # NOTE: Adding the checkpointer for taking snapshots of the conversation as STATES
)

In [20]:
config = {"configurable": {"thread_id": "1"}}

response = agent_stm.invoke(
    {"messages": [question1]},
    config, # NOTE: This is for saving a snapshot of the chats.
)

Markdown(response['messages'][-1].content)

Hi Nirmit! It's so nice to meet you! Blue is a fantastic favorite color. What is it about blue that you love so much? Does it remind you of anything special, like the sky or the ocean? I'm always curious to hear what makes a color someone's favorite! 😊

In [21]:
response = agent_stm.invoke(
    {"messages": [question2]},
    config,  
)

Markdown(response["messages"][-1].content)

Of course I do! You told me your favorite color is **blue**! It's a great choice! 😄

> ##### As we can see here, by adding the thread `config`, the LLM now recalls what we discussed earlier.

In [30]:
query = "Do you still remember my name?"

response = agent_stm.invoke(
    {"messages": [query]},
    {
        "configurable": {
            "thread_id": f"{1}"  # NOTE: Correct ID :::: 0 - ❎ / 1 - ☑️ / 2 - ❎
        }
    },
)

Markdown(response["messages"][-1].content)

Absolutely, Nirmit! Your name is Nirmit.  It's a pleasure to remember it! 😊  So, Nirmit, what's on your mind today?

> ##### It depends upon the `thread_id` parameter. If we do not provide the correct `thread_id`, the LLM will not look up the correct checkpoint for fetching answer(s).